In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Input

In [2]:
# ==========================================
# 1. CREATE A SIMPLE 3-FEATURE DATASET
# ==========================================
np.random.seed(42)
tf.random.set_seed(42)

num_steps = 1000
t = np.linspace(0, 50, num_steps)

# 3 Input Features
feature_1 = np.sin(t) + np.random.normal(0, 0.05, num_steps)        # e.g., Temperature
feature_2 = np.cos(t) + np.random.normal(0, 0.05, num_steps)        # e.g., Humidity
feature_3 = 0.5 * np.sin(2 * t) + np.random.normal(0, 0.05, num_steps) # e.g., Wind Speed

# Combine into shape (1000, 3)
data = np.column_stack((feature_1, feature_2, feature_3))

# Convert into sliding time windows (e.g., look back 10 time steps to predict step 11)
window_size = 10
X, y = [], []
for i in range(len(data) - window_size):
    X.append(data[i : i + window_size])      # Shape: (10, 3) -> 10 steps, 3 features
    y.append(data[i + window_size, 0])       # Target: next value of feature_1

X = np.array(X)  # Final shape: (990, 10, 3)
y = np.array(y)  # Final shape: (990, 1)

In [3]:
# Train / Test Split (80% Train, 20% Test)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# ==========================================
# 2. BUILD THE 2-HIDDEN-LAYER RNN MODEL
# ==========================================
model = Sequential([
    # Input: (time_steps=10, features=3)
    Input(shape=(window_size, 3)),

    # Hidden Layer 1: return_sequences=True is REQUIRED to pass sequences to Hidden Layer 2
    SimpleRNN(units=3, activation='tanh', return_sequences=True, name="Hidden_Layer_1"),

    # Hidden Layer 2: return_sequences=False outputs only the final time step to the Output Layer
    SimpleRNN(units=2, activation='tanh', return_sequences=False, name="Hidden_Layer_2"),

    # Output Layer: 1 neuron
    Dense(units=1, activation='linear', name="Output_Layer")
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()




Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Hidden_Layer_1 (SimpleRNN)      │ (None, 10, 3)          │            21 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_2 (SimpleRNN)      │ (None, 2)              │            12 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 36 (144.00 B)

 Trainable params: 36 (144.00 B)

 Non-trainable params: 0 (0.00 B)

In [4]:
# ==========================================
# 3. TRAIN THE MODEL
# ==========================================
history = model.fit(
    X_train, y_train,
    epochs=25,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)



Epoch 1/25
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - loss: 0.1499 - mae: 0.3026 - val_loss: 0.0442 - val_mae: 0.1659
Epoch 2/25
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0913 - mae: 0.2309 - val_loss: 0.0199 - val_mae: 0.1164
Epoch 3/25
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0626 - mae: 0.2003 - val_loss: 0.0220 - val_mae: 0.1258
Epoch 4/25
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0461 - mae: 0.1786 - val_loss: 0.0213 - val_mae: 0.1251
Epoch 5/25
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0358 - mae: 0.1608 - val_loss: 0.0192 - val_mae: 0.1190
Epoch 6/25
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0289 - mae: 0.1456 - val_loss: 0.0161 - val_mae: 0.1088
Epoch 7/25
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0236 - mae: 0.1313 - val_loss: 0.0133 - val_mae: 0.0979
Epoch 8/25
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0194 - mae: 0.1184 - val_loss: 0.0110 - val_mae: 0.0886
Epoch 9/25
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0161 - mae:

In [5]:

# ==========================================
# 4.TEST THE MODEL
# ==========================================
# Evaluate on unseen Test Data
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest MSE Loss: {test_loss:.4f} | Test MAE: {test_mae:.4f}")

# Sample Predictions vs Actuals
predictions = model.predict(X_test[:5], verbose=0)
for i in range(5):
    print(f"Sample {i+1} -> Actual: {y_test[i]:.4f} | Predicted: {predictions[i][0]:.4f}")


Test MSE Loss: 0.0067 | Test MAE: 0.0626
Sample 1 -> Actual: 0.6495 | Predicted: 0.7172
Sample 2 -> Actual: 0.5825 | Predicted: 0.6926
Sample 3 -> Actual: 0.5433 | Predicted: 0.6002
Sample 4 -> Actual: 0.5076 | Predicted: 0.5650
Sample 5 -> Actual: 0.4909 | Predicted: 0.4680
